## What is cURL?
**cURL** (short for "Client URL") is a command-line tool used to transfer data to or from a server. Think of it as a web browser without the graphical interface—instead of clicking buttons and seeing images, you type commands and get raw data back.

---

### Why Use It?
* **Automation:** Since it’s a command-line tool, you can put it in scripts (Bash, Python, etc.) to automate tasks like uploading logs or checking if a website is down.
* **Platform Independent:** It works on Windows, macOS, and Linux.
* **Protocol Powerhouse:** It supports almost everything—HTTP, HTTPS, FTP, IMAP (email), and even specialized database protocols like InfluxDB's.
* **Debugging:** It’s the "gold standard" for developers to test APIs. If an API works in cURL but not in your code, you know the problem is in your code, not the server.

---

### When to Use It?
* **Testing APIs:** Quickly checking if a server responds correctly to a `GET` or `POST` request.
* **Downloading Files:** Grabbing installers or datasets directly from a terminal.
* **Database Interaction:** Like you’ve been doing—sending data points to InfluxDB using `--data-binary`.
* **IoT & Embedded Systems:** It’s lightweight enough to run on tiny devices that don't have the memory for a full browser.

---

### A Simple "Cheat Sheet"


| Command | What it does |
| :--- | :--- |
| `curl http://example.com` | Fetches the HTML of a page (a basic GET request). |
| `curl -I http://example.com` | Fetches only the **Headers** (metadata like status codes). |
| `curl -o file.zip http://url.com/f.zip` | Downloads a file and saves it as `file.zip`. |
| `curl -u user:pass ...` | Handles **Authentication** (logging in). |
| `curl -X POST -d "data" ...` | Sends a **POST** request with data. |

### Recap
cURL is like the Swiss Army knife you keep in your pocket. It’s not flashy, it’s not "pretty," and if you use it wrong, you might accidentally delete something (or at least get a very angry 404 error). But once you know how to use it, you’ll wonder how you ever managed without it.

Since you've been using it for InfluxDB, are you planning to wrap these commands into a shell script to automate your data uploads?

Let's create a database:
-u means user
-i include the response header in the output, allowing you to see the status code
-XPOST This option sets the HTTP request method to post

/query  The /query endpoint is often used for interacting with InfluxDB data

--data-urlencode tells curl to URL-encode the data string.
q=CREATE DATABASE CURLdb is the actual query string sent to the InfluxDB server. It consists of the keyword CREATE DATABASE followed by the desired database name (CURLdb).

In [ ]:
#!!!!Curl se vedno piše v TERMINAL, to ni python koda!!!!!
curl -u admin:fis_influx -i -XPOST http://149.62.71.186:8086/query --data-urlencode "q=CREATE DATABASE CURLdb"

Let's get a list of existing databases

-G This flag tells curl to use an HTTP GET request. GET requests are typically used to retrieve data from a server. In this case, it's trying to get a list of databases.

?pretty=true part is a query parameter instructing the server to return the response in a more readable format.

q=show databases. This query instructs the InfluxDB server to show existing databases.





In [ ]:
curl -u admin:fis_influx -G "http://149.62.71.186:8086/query?pretty=true" --data-urlencode "q=show databases"

Let's write a data-point to curldb

/write: The endpoint for writing data points to InfluxDB.

?db=CURLdb specifies the database (CURLdb) where you want to insert the data.

--data-binary tells curl to send the data in its raw binary format.

Temperature: The measurement name (field name)
T1: The tag set (key-value pairs describing the data point)
=2: The field value (the actual data)


In [ ]:
curl -u admin:fis_influx -i -XPOST 'http://149.62.71.186:8086/write?db=CURLdb' --data-binary 'Temperature T1=2'

### The Structure of Line Protocol
Each line in your text file represents a single data point and follows this syntax:

`measurement,tag_set field_set timestamp`

---

### How `file1.txt` Should Look
Here is a concrete example of how you should structure the content inside your file:

```text
cpu_load,host=server01,region=us-west value=0.64 1712655886000000000
cpu_load,host=server02,region=us-east value=0.82 1712655886000000000
mem_usage,host=server01 percent_free=12.5 1712655886000000000
```

### Breakdown of the Components
* **Measurement:** (e.g., `cpu_load`) Think of this like a table name in a traditional database.
* **Tag Set:** (e.g., `host=server01,region=us-west`) These are key-value pairs used for metadata. They are **indexed**, so use them for things you want to filter by (like server names or locations). Separate them from the measurement with a **comma**.
* **Field Set:** (e.g., `value=0.64`) These are the actual data values. These are **not indexed**. Separate them from the tags with a **space**.
* **Timestamp:** (Optional) A Unix timestamp in nanoseconds. If you leave this off, InfluxDB will use its current system time when it receives the data.

---

### Important Formatting Rules
1.  **No Spaces in Names:** Do not put spaces between the measurement, commas, and tags. 
    * *Correct:* `weather,location=london temp=25`
    * *Incorrect:* `weather, location=london temp=25`
2.  **Data Types:** * **Floats:** `value=1.0`
    * **Integers:** `value=1i` (Note the **i** suffix)
    * **Strings:** `value="hello"` (Must be in double quotes)
    * **Booleans:** `value=true` or `value=t`
3.  **New Lines:** Each data point must be on its own line. Ensure there isn't a blank line at the very beginning of the file.

> **Tip:** Since you are using `--data-binary`, make sure your file uses Unix-style line endings (`LF`) rather than Windows style (`CRLF`) to avoid unexpected character issues during the POST request.

In [ ]:
INFLUXDB_HOST="149.62.71.186"
INFLUXDB_PORT="8086"
INFLUXDB_USER="admin"
INFLUXDB_PASSWORD="fis_influx"
INFLUXDB_DATABASE="CURLdb"

DATA_FILE="file1.txt"
INFLUXDB_URL="http://${INFLUXDB_HOST}:${INFLUXDB_PORT}/write?db=${INFLUXDB_DATABASE}"
curl -u "${INFLUXDB_USER}:${INFLUXDB_PASSWORD}" -XPOST --data-binary "@${DATA_FILE}" "${INFLUXDB_URL}"

Let's get data from db Naloga 1 and write it to a file

--data-urlencode "db=Naloga 1" This part specifies the database (db=Naloga 1) from which you want to retrieve data.

SELECT "T1" FROM "Temperature": This is the InfluQL query that instructs the server to select the field named "T1" from the measurement named "Temperature".

--output file2.txt

This option tells curl to save the output of the request (the response from the InfluxDB server) to a file named file2.txt.